In [58]:
import pandas as pd
import time
import imdb
import os
import requests
from dotenv import load_dotenv
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [59]:
data = {
    'script_text': [
        "EXT. SPACE - DAY. An explosion rocks the galaxy.",
        "INT. CAFE - NIGHT. They talk about love and coffee.",
        "EXT. MOUNTAIN - DAY. The hero climbs the jagged rocks.",
        "INT. OFFICE - DAY. A boring meeting about spreadsheets.",
        "EXT. BATTLEFIELD - NIGHT. Soldiers charge through the mud."
    ],
    'is_successful': [1, 1, 1, 0, 0]
}

df = pd.DataFrame(data)

In [60]:
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['script_text'])
bow_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
display(bow_df.head())

Vocabulary size: 24


,battlefield,boring,cafe,charge,climbs,coffee,day,explosion,ext,galaxy,...,meeting,mountain,mud,night,office,rocks,soldiers,space,spreadsheets,talk
0,0,0,0,0,0,0,1,1,1,1,...,0,0,0,0,0,1,0,1,0,0
1,0,0,1,0,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
2,0,0,0,0,1,0,1,0,1,0,...,0,1,0,0,0,1,0,0,0,0
3,0,1,0,0,0,0,1,0,0,0,...,1,0,0,0,1,0,0,0,1,0
4,1,0,0,1,0,0,0,0,1,0,...,0,0,1,1,0,0,1,0,0,0


In [61]:
y = df['is_successful']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=67)

baseline_model = LogisticRegression()
baseline_model.fit(X_train, y_train)

word_weights = pd.DataFrame({
    'word': vectorizer.get_feature_names_out(),
    'weight': baseline_model.coef_[0]
}).sort_values(by='weight', ascending=False)

print("Words most associated with success:")
print(word_weights.head(3))

Words most associated with success:
      word    weight
2     cafe  0.264874
5   coffee  0.264874
23    talk  0.264874


In [62]:
movie_titles = [
    "10 Things I Hate About You", "12", "12 and Holding", "12 Monkeys", "12 Years a Slave",
    "127 Hours", "1492: Conquest of Paradise", "15 Minutes", "17 Again", "187",
    "2001: A Space Odyssey", "2012", "25th Hour", "30 Minutes or Less", "42",
    "44 Inch Chest", "48 Hrs.", "50-50", "500 Days of Summer", "8MM", "9",
    "A Few Good Men", "A Serious Man", "Above the Law", "Absolute Power", "Abyss, The",
    "Ace Ventura: Pet Detective", "Adaptation", "Addams Family, The", "Adjustment Bureau, The",
    "Adventures of Buckaroo Banzai Across the Eighth Dimension, The", "Affliction",
    "After School Special", "After.Life", "Agnes of God", "Air Force One", "Airplane",
    "Airplane 2: The Sequel", "Aladdin", "Ali", "Alien 3", "Alien Nation", "Alien vs. Predator",
    "Alien: Resurrection", "Aliens", "All About Eve", "All About Steve", "All the King's Men",
    "All the President's Men", "Almost Famous", "Alone in the Dark", "Amadeus", "Amelia",
    "American Beauty", "American Gangster", "American Graffiti", "American History X",
    "American Hustle", "American Madness", "American Outlaws", "American Pie",
    "American President, The", "American Psycho", "American Shaolin: King of Kickboxers II",
    "American Splendor", "American Werewolf in London", "American, The", "Amityville Asylum, The",
    "Amour", "An Education", "Analyze That", "Analyze This", "Anastasia", "Angel Eyes",
    "Angels & Demons", "Anna Karenina", "Annie Hall", "Anniversary Party, The", "Anonymous",
    "Antitrust", "Antz", "Apartment, The", "Apocalypse Now", "April Fool's Day", "Apt Pupil",
    "Arbitrage", "Arcade", "Arctic Blue", "Argo", "Armageddon", "Army of Darkness",
    "Arsenic and Old Lace", "Arthur", "Artist, The", "As Good As It Gets", "Assassins",
    "Assignment, The", "At First Sight", "August: Osage County", "Austin Powers - International Man of Mystery",
    "Austin Powers - The Spy Who Shagged Me", "Authors Anonymous", "Autumn in New York", "Avatar",
    "Avengers, The", "Avengers, The (2012)", "Avventura, L' (The Adventure)", "Awakenings",
    "Babel", "Bachelor Party", "Bachelor Party, The", "Back-up Plan, The", "Backdraft", "Bad Boys",
    "Bad Country", "Bad Day at Black Rock", "Bad Dreams", "Bad Lieutenant", "Bad Santa",
    "Bad Teacher", "Badlands", "Bamboozled", "Barry Lyndon", "Barton Fink", "Basic",
    "Basic Instinct", "Basquiat", "Batman", "Batman 2", "Battle of Algiers, The",
    "Battle of Shaker Heights, The", "Battle: Los Angeles", "Beach, The", "Bean",
    "Beasts of the Southern Wild", "Beavis and Butt-head Do America", "Beginners", "Being Human",
    "Being John Malkovich", "Being There", "Believer, The", "Beloved", "Benny & Joon",
    "Best Exotic Marigold Hotel, The", "Big", "Big Blue, The", "Big Fish", "Big Lebowski, The",
    "Big White, The", "Birds, The", "Birthday Girl", "Black Dahlia, The", "Black Rain",
    "Black Snake Moan", "Black Swan", "Blade", "Blade II", "Blade Runner", "Blade: Trinity",
    "Blast from the Past, The", "Blind Side, The", "Bling Ring, The", "Blood and Wine",
    "Blood Simple", "Blow", "Blue Valentine", "Blue Velvet", "Bodies, Rest & Motion", "Body Heat",
    "Body of Evidence", "Bodyguard", "Bones", "Bonfire of the Vanities", "Bonnie and Clyde",
    "Boogie Nights", "Book of Eli, The", "Boondock Saints 2: All Saints Day", "Boondock Saints, The",
    "Bottle Rocket", "Bound", "Bounty Hunter, The", "Bourne Identity, The", "Bourne Supremacy, The",
    "Bourne Ultimatum, The", "Box, The", "Braveheart", "Brazil", "Break", "Breakdown",
    "Breakfast Club, The", "Breaking Away", "Brick", "Bridesmaids", "Bringing Out the Dead",
    "Broadcast News", "Broken Arrow", "Broken Embraces", "Brothers Bloom, The", "Bruce Almighty",
    "Buffy the Vampire Slayer", "Bull Durham", "Buried", "Burlesque", "Burn After Reading",
    "Burning Annie", "Butterfly Effect, The", "Cable Guy", "Candle to Water", "Capote", "Carrie",
    "Cars 2", "Case 39", "Casino", "Cast Away", "Catch Me If You Can", "Catwoman", "Cecil B. Demented",
    "Cedar Rapids", "Celeste & Jesse Forever", "Cell, The", "Cellular", "Change-Up, The",
    "Changeling", "Chaos", "Charade", "Charlie's Angels", "Chasing Amy", "Chasing Sleep",
    "Cherry Falls", "Chinatown", "Christ Complex", "Chronicle",
    "Chronicles of Narnia: The Lion, the Witch and the Wardrobe", "Cider House Rules, The",
    "Cincinnati Kid, The", "Cinema Paradiso", "Cirque du Freak: The Vampire's Assistant",
    "Citizen Kane", "City of Joy", "Clash of the Titans", "Clerks", "Cliffhanger", "Clueless", "Cobb",
    "Code of Silence", "Cold Mountain", "Collateral", "Collateral Damage", "Colombiana",
    "Color of Night", "Commando", "Conan the Barbarian", "Confessions of a Dangerous Mind",
    "Confidence", "Constantine", "Cooler, The", "Copycat", "Coraline", "Coriolanus",
    "Cowboys & Aliens", "Cradle 2 the Grave", "Crank", "Crash", "Crazy, Stupid, Love", "Crazylove",
    "Creation", "Crime Spree", "Croods, The", "Crouching Tiger, Hidden Dragon", "Croupier",
    "Crow Salvation, The", "Crow, The", "Crow: City of Angels, The", "Cruel Intentions", "Crying Game",
    "Cube", "Curious Case of Benjamin Button, The", "Custody", "Damned United, The",
    "Dances with Wolves", "Dark City", "Dark Knight Rises, The", "Dark Star", "Darkman", "Date Night",
    "Dave Barry's Complete Guide to Guys", "Dawn of the Dead", "Day of the Dead",
    "Day the Clown Cried, The", "Day the Earth Stood Still, The", "Days of Heaven",
    "Dead Poets Society", "Death at a Funeral", "Death to Smoochy", "Debt, The", "Deception",
    "Deep Cover", "Deep Rising", "Deer Hunter, The", "Defiance", "Departed, The", "Descendants, The",
    "Despicable Me 2", "Detroit Rock City", "Devil in a Blue Dress", "Devil's Advocate", "Die Hard",
    "Die Hard 2", "Diner", "Distinguished Gentleman, The", "Disturbia", "Django Unchained",
    "Do The Right Thing", "Dog Day Afternoon", "Dogma", "Donnie Brasco", "Doors, The",
    "Double Indemnity", "Drag Me to Hell", "Dragonslayer", "Drive", "Drive Angry",
    "Drop Dead Gorgeous", "Dry White Season, A", "Duck Soup", "Dumb and Dumber", "Dune", "E.T.",
    "Eagle Eye", "Eastern Promises", "Easy A", "Ed TV", "Ed Wood", "Edward Scissorhands",
    "Eight Legged Freaks", "El Mariachi", "Election", "Elephant Man, The", "Elizabeth: The Golden Age",
    "Enemy of the State", "English Patient, The", "Enough", "Entrapment", "Erik the Viking",
    "Erin Brockovich", "Escape From L.A.", "Escape From New York",
    "Eternal Sunshine of the Spotless Mind", "Even Cowgirls Get the Blues", "Event Horizon",
    "Evil Dead", "Evil Dead II: Dead by Dawn", "Excalibur", "eXistenZ", "Extract",
    "Fabulous Baker Boys, The", "Face Off", "Fair Game", "Family Man, The", "Fantastic Four",
    "Fantastic Mr Fox", "Fargo", "Fast Times at Ridgemont High", "Fatal Instinct",
    "Fault in Our Stars, The", "Fear and Loathing in Las Vegas", "Feast", "Ferris Bueller's Day Off",
    "Field of Dreams", "Fifth Element, The", "Fight Club", "Fighter, The", "Final Destination",
    "Final Destination 2", "Finding Nemo", "Five Easy Pieces", "Flash Gordon", "Fletch", "Flight",
    "Flintstones, The", "Forrest Gump", "Four Feathers", "Four Rooms", "Foxcatcher", "Fracture",
    "Frances", "Frankenstein", "Freaked", "Freddy vs. Jason", "French Connection, The", "Frequency",
    "Friday the 13th", "Friday the 13th Part VIII: Jason Takes Manhattan", "Fright Night",
    "Fright Night (1985)", "From Dusk Till Dawn", "From Here to Eternity", "Frozen",
    "Frozen (Disney)", "Frozen River", "Fruitvale Station", "Fugitive, The", "Funny People",
    "G.I. Jane", "G.I. Joe: The Rise of Cobra", "Game 6", "Game, The", "Gamer", "Gandhi",
    "Gang Related", "Gangs of New York", "Garden State", "Gattaca", "Get Carter", "Get Low",
    "Get Shorty", "Getaway, The", "Ghost", "Ghost and the Darkness, The", "Ghost Rider",
    "Ghost Ship", "Ghost World", "Ghostbusters", "Ghostbusters 2", "Ginger Snaps",
    "Girl with the Dragon Tattoo, The", "Gladiator", "Glengarry Glen Gross", "Go", "Godfather",
    "Godfather Part II", "Godfather Part III, The", "Gods and Monsters", "Godzilla",
    "Gone in 60 Seconds", "Good Girl, The", "Good Will Hunting", "Gothika", "Graduate, The",
    "Gran Torino", "Grand Hotel", "Grand Theft Parsons", "Grapes of Wrath, The", "Gravity",
    "Green Mile, The", "Gremlins", "Gremlins 2", "Grifters, The", "Grosse Point Blank",
    "Groundhog Day", "Grudge, The", "Hackers", "Hall Pass", "Halloween: The Curse of Michael Myers",
    "Hancock", "Hangover, The", "Hanna", "Hannah and Her Sisters", "Hannibal",
    "Happy Birthday, Wanda June", "Happy Feet", "Hard Rain", "Hard to Kill",
    "Harold and Kumar Go to White Castle", "Haunting, The", "He's Just Not That Into You", "Heat",
    "Heathers", "Heavenly Creatures", "Heavy Metal", "Hebrew Hammer, The", "Heist",
    "Hellbound: Hellraiser II", "Hellboy", "Hellboy 2: The Golden Army", "Hellraiser",
    "Hellraiser 3: Hell on Earth", "Hellraiser: Deader", "Hellraiser: Hellseeker", "Help, The",
    "Henry Fool", "Henry's Crime", "Hesher", "High Fidelity", "Highlander", "Highlander: Endgame",
    "Hills Have Eyes, The", "His Girl Friday", "Hitchcock", "Hitchhiker's Guide to the Galaxy, The",
    "Hollow Man", "Honeydripper", "Horrible Bosses", "Horse Whisperer, The", "Hospital, The",
    "Hostage", "Hot Tub Time Machine", "Hotel Rwanda", "House of 1000 Corpses",
    "How to Lose Friends & Alienate People", "How to Train Your Dragon", "How to Train Your Dragon 2",
    "Hudson Hawk", "Hudsucker Proxy, The", "Human Nature", "Hunt for Red October, The",
    "I Am Number Four", "I am Sam", "I Love You Phillip Morris", "I Spit on Your Grave",
    "I Still Know What You Did Last Summer", "I'll Do Anything", "I, Robot", "Ice Storm, The",
    "Ides of March, The", "Imaginarium of Doctor Parnassus, The", "In the Bedroom", "In the Loop",
    "Inception", "Incredibles, The", "Independence Day", "Indiana Jones and the Last Crusade",
    "Indiana Jones and the Raiders of the Lost Ark", "Indiana Jones and the Temple of Doom",
    "Indiana Jones IV", "Informant, The", "Inglourious Basterds", "Insider, The", "Insidious",
    "Insomnia", "Interstellar", "Interview with the Vampire", "Into the Wild", "Intolerable Cruelty",
    "Inventing the Abbotts", "Invention of Lying, The", "Invictus", "Iron Lady, The", "Island, The",
    "It Happened One Night", "It's a Wonderful Life", "It's Complicated", "Italian Job, The",
    "Jacket, The", "Jackie Brown", "Jacob's Ladder", "Jane Eyre", "Jason X", "Jaws", "Jaws 2",
    "Jay and Silent Bob Strike Back", "Jennifer Eight", "Jennifer's Body", "Jerry Maguire", "JFK",
    "Jimmy and Judy", "John Q", "Judge Dredd", "Juno", "Jurassic Park", "Jurassic Park III",
    "Jurassic Park: The Lost World", "Kafka", "Kalifornia", "Kate & Leopold", "Kids",
    "Kids Are All Right, The", "Kill Bill Volume 1 & 2", "Killing Zoe", "King Kong",
    "King of Comedy, The", "King's Speech, The", "Kingdom, The", "Klute", "Knocked Up",
    "Kramer vs Kramer", "Kundun", "Kung Fu Panda", "L.A. Confidential", "Labor of Love", "Labyrinth",
    "Ladykillers, The", "Lake Placid", "Land of the Dead", "Larry Crowne", "Last Boy Scout, The",
    "Last Chance Harvey", "Last Flight, The", "Last of the Mohicans, The", "Last Samurai, The",
    "Last Station, The", "Last Tango in Paris", "Law Abiding Citizen", "Leaving Las Vegas",
    "Legally Blonde", "Legend", "Legion", "Les Miserables", "Leviathan", "Liar Liar", "Life",
    "Life As A House", "Life of David Gale, The", "Life of Pi", "Light Sleeper", "Limey, The",
    "Limitless", "Lincoln", "Lincoln Lawyer, The", "Little Athens", "Little Mermaid, The",
    "Little Nicky", "Living in Oblivion", "Lock, Stock and Two Smoking Barrels", "Logan's Run",
    "Lone Star", "Long Kiss Goodnight, The", "Looper", "Lord of Illusions",
    "Lord of the Rings: Fellowship of the Ring, The", "Lord of the Rings: Return of the King",
    "Lord of the Rings: The Two Towers", "Lord of War", "Losers, The", "Lost Highway",
    "Lost Horizon", "Lost in Space", "Lost in Translation", "Love and Basketball", "Machete",
    "Machine Gun Preacher", "Mad Max 2: The Road Warrior", "Made", "Magnolia",
    "Majestic, The (The Bijou)", "Major League", "Malcolm X", "Malibu's Most Wanted",
    "Man in the Iron Mask", "Man On Fire", "Man on the Moon", "Man Trouble",
    "Man Who Knew Too Much, The", "Man Who Wasn't There, The", "Manchurian Candidate, The",
    "Manhattan Murder Mystery", "Manhunter", "Margaret", "Margin Call", "Margot at the Wedding",
    "Mariachi, El", "Marley & Me", "Martha Marcy May Marlene", "Marty", "Mary Poppins", "Mask, The",
    "Master and Commander", "Master, The", "Matrix Reloaded, The", "Matrix, The", "Max Payne",
    "Mean Streets", "Mechanic, The", "Meet Joe Black", "Meet John Doe", "Megamind", "Memento",
    "Men in Black", "Men in Black 3", "Men Who Stare at Goats, The", "Metro", "Miami Vice",
    "Midnight Cowboy", "Midnight Express", "Midnight in Paris",
    "Mighty Morphin Power Rangers: The Movie", "Milk", "Miller's Crossing", "Mimic",
    "Mini's First Time", "Minority Report", "Miracle Worker, The", "Mirrors", "Misery",
    "Mission Impossible", "Mission Impossible II", "Mission to Mars", "Moneyball", "Monkeybone",
    "Monte Carlo", "Moon", "Moonrise Kingdom", "Moonstruck", "Mr Blandings Builds His Dream House",
    "Mr Brooks", "Mr Deeds Goes to Town", "Mrs. Brown", "Mud", "Mulan", "Mulholland Drive",
    "Mumford", "Mummy, The", "Music of the Heart", "Mute Witness", "My Best Friend's Wedding",
    "My Girl", "My Mother Dreams the Satan's Disciples in New York", "My Week with Marilyn",
    "Mystery Men", "Nashville", "Natural Born Killers", "Never Been Kissed", "Neverending Story, The",
    "New York Minute", "Newsies", "Next", "Next Friday", "Next Three Days, The", "Nick of Time",
    "Night Time (The Poltergeist Treatment)", "Nightbreed", "Nightmare Before Christmas, The",
    "Nightmare on Elm Street, A", "Nightmare on Elm Street: The Final Chapter", "Nine", "Nines, The",
    "Ninja Assassin", "Ninotchka", "Ninth Gate, The", "No Strings Attached", "Notting Hill",
    "Nurse Betty", "O Brother Where Art Thou?", "Oblivion", "Observe and Report", "Obsessed",
    "Ocean's Eleven", "Ocean's Twelve", "Office Space", "Omega Man", "One Flew Over the Cuckoo's Nest",
    "Only God Forgives", "Ordinary People", "Orgy of the Dead", "Orphan", "Other Boleyn Girl, The",
    "Out of Sight", "Pacifier, The", "Pandorum", "Panic Room", "Papadopoulos & Sons", "ParaNorman",
    "Pariah", "Passion of Joan of Arc, The", "Patriot, The", "Paul", "Pearl Harbor", "Peeping Tom",
    "Peggy Sue Got Married", "Perfect Creature", "Perfect World, A", "Perks of Being a Wallflower, The",
    "Pet Sematary", "Pet Sematary II", "Petulia", "Philadelphia", "Phone Booth", "Pi", "Pianist, The",
    "Piano, The", "Pineapple Express", "Pirates of the Caribbean",
    "Pirates of the Caribbean: Dead Man's Chest", "Pitch Black", "Planet of the Apes, The",
    "Platinum Blonde", "Platoon", "Pleasantville", "Point Break", "Pokemon: Mewtwo Returns",
    "Postman, The", "Power of One, The", "Precious", "Predator", "Pretty Woman",
    "Pride and Prejudice", "Priest", "Princess Bride, The", "Private Life of Sherlock Holmes, The",
    "Producer, The", "Program, The", "Prom Night", "Prometheus", "Prophecy, The", "Proposal, The",
    "Psycho", "Public Enemies", "Pulp Fiction", "Punch-Drunk Love", "Purple Rain", "Quantum Project",
    "Queen of the Damned", "Queen, The", "Rachel Getting Married", "Raging Bull", "Raising Arizona",
    "Rambling Rose", "Rambo: First Blood II: The Mission", "Reader, The", "Real Genius",
    "Rear Window", "Rebel Without A Cause", "Red Planet", "Red Riding Hood", "Reindeer Games",
    "Relic, The", "Remember Me", "Replacements, The", "Repo Man", "Rescuers Down Under, The",
    "Reservoir Dogs", "Resident Evil", "Return of the Apes", "Revenant, The", "Revolutionary Road",
    "Ringu", "Rise of the Guardians", "Rise of the Planet of the Apes", "RKO 281", "Road, The",
    "Robin Hood: Prince of Thieves", "Rock, The", "RocknRolla", "Rocky",
    "Rocky Horror Picture Show, The", "Romeo & Juliet", "Ronin", "Roommate, The", "Roughshod",
    "Ruins, The", "Runaway Bride", "Rush", "Rush Hour", "Rush Hour 2", "Rushmore", "Rust and Bone",
    "S. Darko", "Saint, The", "Salton Sea, The", "Sandlot Kids, The", "Save the Last Dance",
    "Saving Mr. Banks", "Saving Private Ryan", "Saw", "Scarface", "Schindler's List",
    "Scott Pilgrim vs the World", "Scream", "Scream 2", "Scream 3", "Se7en", "Searchers, The",
    "Semi-Pro", "Sense and Sensibility", "Serenity", "Serial Mom", "Sessions, The",
    "Seventh Seal, The", "Sex and the City", "Sex, Lies and Videotape", "Sexual Life",
    "Shakespeare in Love", "Shallow Grave", "Shame", "Shampoo", "Shawshank Redemption, The",
    "She's Out of My League", "Sherlock Holmes", "Shifty", "Shining, The", "Shipping News, The",
    "Shivers", "Shrek", "Shrek the Third", "Sideways", "Siege, The", "Signs", "Silence of the Lambs",
    "Silver Bullet", "Silver Linings Playbook", "Simone", "Single White Female", "Sister Act",
    "Six Degrees of Separation", "Sixth Sense, The", "Sleepless in Seattle", "Sleepy Hollow",
    "Sling Blade", "Slither", "Slumdog Millionaire", "Smashed", "Smokin' Aces", "Snatch",
    "Snow Falling On Cedars", "Snow White and the Huntsman", "So I Married an Axe Murderer",
    "Social Network, The", "Solaris", "Soldier", "Someone To Watch Over Me", "Something's Gotta Give",
    "Source Code", "South Park", "Spanglish", "Spare Me", "Spartan", "Speed Racer", "Sphere",
    "Spider-Man", "St. Elmo's Fire", "Star Trek", "Star Trek II: The Wrath of Khan",
    "Star Trek: First Contact", "Star Trek: Generations", "Star Trek: Nemesis",
    "Star Trek: The Motion Picture", "Star Wars: A New Hope", "Star Wars: Attack of the Clones",
    "Star Wars: Return of the Jedi", "Star Wars: Revenge of the Sith",
    "Star Wars: The Empire Strikes Back", "Star Wars: The Force Awakens",
    "Star Wars: The Phantom Menace", "Starman", "Starship Troopers", "State and Main",
    "Station West", "Stepmom", "Sting, The", "Stir of Echoes", "Storytelling", "Strange Days",
    "Strangers on a Train", "Stuntman, The", "Sugar", "Sugar and Spice", "Sunset Blvd.",
    "Sunshine Cleaning", "Super 8", "Superbad", "Supergirl", "Surfer King, The", "Surrogates",
    "Suspect Zero", "Sweeney Todd: The Demon Barber of Fleet Street", "Sweet Hereafter, The",
    "Sweet Smell of Success", "Swingers", "Swordfish", "Synecdoche, New York", "Syriana",
    "Take Shelter", "Taking Lives", "Taking of Pelham One Two Three, The", "Taking Sides",
    "Talented Mr. Ripley, The", "Tall in the Saddle", "Tamara Drewe", "Taxi Driver", "Ted",
    "Terminator", "Terminator 2: Judgement Day", "Terminator Salvation", "The Rage: Carrie 2",
    "Thelma & Louise", "There's Something About Mary", "They", "Thing, The",
    "Things My Father Never Taught Me, The", "Thirteen Days", "This Boy's Life", "This is 40",
    "Thor", "Three Kings", "Three Kings (Spoils of War)", "Three Men and a Baby",
    "Three Musketeers, The", "Thunderbirds", "Thunderheart", "Ticker", "Timber Falls",
    "Time Machine, The", "Tin Cup", "Tin Men", "Tinker Tailor Soldier Spy", "Titanic", "TMNT",
    "To Sleep with Anger", "Tombstone", "Tomorrow Never Dies", "Top Gun", "Total Recall",
    "Tourist, The", "Toy Story", "Traffic", "Training Day", "Trainspotting",
    "Transformers: The Movie", "Tremors", "Tristan and Isolde", "TRON", "TRON: Legacy",
    "Tropic Thunder", "True Grit", "True Lies", "True Romance", "Truman Show, The", "Twilight",
    "Twilight: New Moon", "Twin Peaks", "Twins", "Two For The Money", "U Turn", "Ugly Truth, The",
    "Unbreakable", "Under Fire", "Unknown", "Up", "Up in the Air", "Usual Suspects, The",
    "V for Vendetta", "Valkyrie", "Vanilla Sky", "Verdict, The", "Very Bad Things", "Village, The",
    "Virtuosity", "Visitor, The", "Wag the Dog", "Walk to Remember, A", "Walking Tall", "Wall Street",
    "Wall Street: Money Never Sleeps", "Wall-E", "Wanted", "War Horse", "War of the Worlds",
    "Warm Springs", "Warrior", "Watchmen", "Water for Elephants", "Way Back, The", "We Own the Night",
    "What About Bob?", "What Lies Beneath", "When a Stranger Calls", "While She Was Out",
    "Whistleblower, The", "White Christmas", "White Jazz", "White Ribbon, The", "White Squall",
    "Whiteout", "Who Framed Roger Rabbit?", "Who's Your Daddy", "Wild At Heart", "Wild Bunch, The",
    "Wild Hogs", "Wild Things", "Wild Things: Diamonds in the Rough", "Wild Wild West", "Willow",
    "Win Win", "Wind Chill", "Withnail and I", "Witness", "Wizard of Oz, The",
    "Wolf of Wall Street, The", "Wonder Boys", "Woodsman, The", "World is not Enough, The",
    "Wrestler, The", "X-Files: Fight the Future, The", "X-Men", "X-Men Origins: Wolverine", "xXx",
    "Year One", "Yes Man", "You Can Count On Me", "You've Got Mail", "Youth in Revolt",
    "Zero Dark Thirty", "Zerophilia"
]

In [ ]:
load_dotenv()

API_KEY = os.getenv('OMDB_API_KEY')

def clean_title_v2(title):
    t = title.replace('"', '').strip()
    if ", The" in t:
        t = "The " + t.replace(", The", "").strip()
    return t

def get_omdb_rating(title):
    if not API_KEY:
        return None
    
    search_term = clean_title_v2(title)
    url = f"http://www.omdbapi.com/?t={search_term}&apikey={API_KEY}"
    
    try:
        response = requests.get(url).json()
        if response.get('Response') == 'True':
            return response.get('imdbRating')
    except:
        return None
    return None

results = []

for title in tqdm(movie_titles):
    rating = get_omdb_rating(title)
    
    try:
        numeric_rating = float(rating) if rating and rating != "N/A" else None
    except ValueError:
        numeric_rating = None
        
    results.append({
        'original_title': title.replace('"', ''),
        'rating': numeric_rating
    })

df = pd.DataFrame(results)
df['is_successful'] = df['rating'].apply(lambda x: 1 if pd.notnull(x) and x >= 7.0 else 0)
df = df[df['rating'].notna()]

df.to_csv('movie_ratings_final.csv', index=False, quoting=1)

print(f"\nFinished! Matched {df['rating'].notnull().sum()} out of {len(movie_titles)} movies.")
print(df.head(10))

Test Rating: 7.4


100%|██████████| 1093/1093 [00:48<00:00, 22.73it/s]


Finished! Matched 263 out of 1093 movies.
               original_title  rating  is_successful
0  10 Things I Hate About You     NaN              0
1                          12     NaN              0
2              12 and Holding     NaN              0
3                  12 Monkeys     NaN              0
4            12 Years a Slave     NaN              0
5                   127 Hours     7.5              1
6  1492: Conquest of Paradise     6.4              0
7                  15 Minutes     NaN              0
8                    17 Again     6.4              0
9                         187     NaN              0


In [64]:
ia = imdb.Cinemagoer()
movies = ia.search_movie("matrix")
print(movies)

[]
